## Matrix Solver Reference Implementation

### Floating Point Test

In [ ]:
from ieee754 import double, IEEE754

# 直接获取双精度表示
print(double(13.375))

# 使用类获取更多信息
x = 13.375
a = IEEE754(x,precision=1) # 默认双精度
print(str(a)) # 二进制表示
print(a.hex()) # 十六进制表示
print(a.json()) # 详细信息(JSON)

0 10000000010 1010110000000000000000000000000000000000000000000000
0 10000010 10101100000000000000000
('41560000', ['0100', '0001', '0101', '0110', '0000', '0000', '0000', '0000'])
{'number': Decimal('13.375'), 'edge_case': False, 'sign_bit': '1', 'exponent_bits': 8, 'mantissa_bits': 23, 'total_bits': 32, 'sign': '0', 'scale': 3, 'scaled_number': 107, 'scaled_number_in_binary': '1101011', 'unable_to_scale': False, 'bias': 127, 'bias_in_binary': '01111111', 'exponent': '10000010', 'mantissa': '10101100000000000000000', 'mantissa_base_10': Decimal('0.67187500000000000000000'), 'result': '0 10000010 10101100000000000000000', 'hexadecimal': '41560000', 'hexadecimal_parts': ['0100', '0001', '0101', '0110', '0000', '0000', '0000', '0000'], 'converted_number': Decimal('13.375'), 'error': Decimal('0.000')}


In [3]:
a, b = 6.25,13.25
a_plus_b = a+b
aa = IEEE754(a, precision=1)
bb = IEEE754(b, precision=1)
rr = IEEE754(a_plus_b, precision=1)
print(f"A        hex : {aa.hex()[0]}")
print(f"B        hex : {bb.hex()[0]}")
print(f"Expected hex : {rr.hex()[0]}")

A        hex : 40C80000
B        hex : 41540000
Expected hex : 419C0000


In [55]:
a, b, c = 122.04, 0.06152, 0.237
r = a * b + c
aa = IEEE754(a, precision=1)
bb = IEEE754(b, precision=1)
cc = IEEE754(c, precision=1)
rr = IEEE754(r, precision=1)
print(f"A        hex : {aa.hex()[0]}")
print(f"B        hex : {bb.hex()[0]}")
print(f"C        hex : {cc.hex()[0]}")
print(f"Expected hex : {rr.hex()[0]}")
print(r)
print(a/b)

A        hex : 42F4147B
B        hex : 3D7BFC65
C        hex : 3E72B021
Expected hex : 40F7D63B
7.7449008
1983.7451235370613


### LU Decomposition

In [ ]:
import numpy as np

# available floating-point primitives

cycles = 0

# cycle: 0
def neg(v: float) -> float:
    return -v

# cycle: 16
def fma(a: float, b: float, c: float) -> float:
    global cycles
    cycles += 16
    return a * b + c

# cycle: 28
def div(a: float, b: float) -> float:
    global cycles
    cycles += 28
    return a / b

def lu_decomp(A: np.ndarray):
    n = A.shape[0]
    # L = np.eye(n, dtype=float)
    # U = np.zeros((n, n), dtype=float)
    L = np.random.rand(n, n)
    U = np.random.rand(n, n)
    for j in range(n):
        for i in range(n):
            if i <= j:
                U[i, j] = neg(A[i, j])
                for k in range(i):
                    U[i, j] = fma(L[i, k], U[k, j], U[i, j]) # k < i
                U[i, j] = neg(U[i, j])
            else:  # i > j
                L[i, j] = neg(A[i, j])
                for k in range(j):
                    L[i, j] = fma(L[i, k] , U[k, j], L[i,j]) # k < j; i > j => k < i
                L[i, j] = div(L[i, j], U[j, j]) 
                L[i, j] = neg(L[i, j])
    return L, U


def lu_decomp_more_hard(A: np.ndarray):
    n = A.shape[0]
    LU = np.random.rand(n, n)

    def fetch_A(i,j):
        return A[i,j]

    def fetch_LU(i, j):
        return LU[i, j]
    
    def store_LU(i,j,v):
        LU[i,j] = v

    for j in range(n):
        for i in range(n):
            if i <= j:
                r1 = fetch_A(i,j)
                r1 = -r1
                for k in range(i):
                    r2 = fetch_LU(i, k)
                    r3 = fetch_LU(k, j)
                    r1 = fma(r2,r3,r1)
                r1 = -r1
                store_LU(i,j,r1)
            else:
                r1 = fetch_A(i, j)
                r1 = -r1
                LU[i, j] = neg(A[i, j])
                for k in range(j):
                    r2 = fetch_LU(i, k)
                    r3 = fetch_LU(k, j)
                    r1 = fma(r2, r3, r1)
                    # LU[i, j] = fma(LU[i, k], LU[k, j], LU[i, j])
                r2 = fetch_LU()
                LU[i, j] = div(LU[i, j], LU[j, j])
                LU[i, j] = -LU[i, j]
    return LU

In [ ]:
cycles = 0; mat = np.random.rand(64, 64); lu_decomp(mat); print(f"64x64 cycles={cycles}")
cycles = 0; mat = np.random.rand(32, 32); lu_decomp(mat); print(f"32x32 cycles={cycles}")
cycles = 0; mat = np.random.rand(16, 16); lu_decomp(mat); print(f"16x16 cycles={cycles}")
cycles = 0; mat = np.random.rand(8, 8); lu_decomp(mat); print(f"8x8 cycles={cycles}")

64x64 cycles=1421952
32x32 cycles=180544
16x16 cycles=23200
8x8 cycles=3024


In [71]:
mat = np.random.rand(4, 4)


In [128]:

lu_decomp(mat)

(array([[ 0.4909371 ,  0.94059478,  0.95149571,  0.56955022],
        [ 2.83810365,  0.10905691,  0.5177227 ,  0.2293237 ],
        [ 1.25494354,  0.48685072,  0.86437973,  0.40644642],
        [ 0.89196089,  0.09935908, -0.1195219 ,  0.25069679]]),
 array([[ 0.23391204,  0.83237664,  0.74763341,  0.84082188],
        [ 0.62840906, -1.92088819, -1.57184083, -1.93737192],
        [ 0.65613906,  0.41127792,  0.55243059,  0.6593814 ],
        [ 0.95584867,  0.07305298,  0.14471387,  0.15513549]]))

In [137]:
lu_decomp_more_hard(mat)

array([[ 0.23391204,  0.83237664,  0.74763341,  0.84082188],
       [ 2.83810365, -1.92088819, -1.57184083, -1.93737192],
       [ 1.25494354,  0.48685072,  0.55243059,  0.6593814 ],
       [ 0.89196089,  0.09935908, -0.1195219 ,  0.15513549]])